In [8]:
from utils.load_results import *
from utils.plot_helpers import *
from utils.analysis_from_interaction import *


import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
plt.style.use('default')
import torch
from language_analysis_local import TopographicSimilarityConceptLevel, encode_target_concepts_for_topsim
import os
if not os.path.exists('analysis'):
    os.makedirs('analysis')
#import plotly.express as px
from collections import Counter

### Utilities

In [9]:
def objects_to_concepts(sender_input, n_values):
    """reconstruct concepts from objects in interaction"""
    n_targets = int(sender_input.shape[1]/2)
    # get target objects and fixed vectors to re-construct concepts
    target_objects = sender_input[:, :n_targets]
    target_objects = k_hot_to_attributes(target_objects, n_values)
    # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
    (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
    concepts = list(zip(objects, fixed))
    return concepts

In [10]:
def retrieve_messages(interaction):
    """retrieve messages from interaction"""
    messages = interaction.message.argmax(dim=-1)
    messages = [msg.tolist() for msg in messages]
    return messages

In [11]:
def count_symbols(messages):
    """counts symbols in messages"""
    all_symbols = [symbol for message in messages for symbol in message]
    symbol_counts = Counter(all_symbols)
    return symbol_counts

In [12]:
def get_unique_message_set(messages):
    """returns unique messages as a set ready for set operations"""
    return set(tuple(message) for message in messages)

In [13]:
def get_unique_concept_set(concepts):
    """returns unique concepts"""
    concept_tuples = []
    for objects, fixed in concepts:
        tuple_objects = []
        for object in objects:
            tuple_objects.append(tuple(object))
        tuple_objects = tuple(tuple_objects)
        tuple_concept = (tuple_objects, tuple(fixed))
        concept_tuples.append(tuple_concept)
    tuple(concept_tuples)
    unique_concepts = set(concept_tuples)
    return unique_concepts

In [14]:
def look_up_values(index_vector, value_vector, dictionary):
    """
    Look up values in a dictionary for index-attribute pairs from two vectors.

    Args:
        index_vector (list): A list of indices.
        value_vector (list): A list of values corresponding to the indices.
        dictionary (dict): A dictionary with index-attribute pairs as keys.

    Returns:
        list: A list of looked-up values.
    """
    # Initialize an empty list to store the looked-up values
    looked_up_values = []

    # Iterate over the index and value pairs
    for index, value in zip(index_vector, value_vector):
        # Construct the key by concatenating the index and value as strings
        key = f"{index}{value}"

        # Look up the value in the dictionary and append it to the list
        looked_up_values.append(dictionary.get(key))

    return looked_up_values

### Configurations

In [15]:
datasets = ['3dshapes']
n_values = [4]
n_attributes = [3]
n_epochs = 300
n_runs = 3
n_datasets = len(datasets)

paths = [
    'results/3dshapes/shapes3d_feat_rep_game_size_10_vsf_3',  # path for 3dshapes
]

In [336]:
# datasets = ['3dshapes', '(3,4)']
# n_values = [4, 4]
# n_attributes = [3, 3]
# vocab_sizes = 16
# n_epochs = 300
# n_datasets = len(datasets)
# n_runs = 3
# paths = [
#     'results/3dshapes/shapes3d_feat_rep_game_size_10_vsf_3',  # path for 3dshapes
#     'results/(3,4)_game_size_10_vsf_3'                        # path for symbolic (3,4)
# ]

In [16]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'specific' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [26]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'generic' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

### Determine vocab size and message reuse

In [17]:
# Determine vocab size and message reuse (3dshapes only: to specific)

# Loop through all 3dshapes datasets
for i, d in enumerate(datasets):
    print("Dataset:", d)
    
    for run in range(n_runs):
        
        path_to_run = paths[i] + '/' + str(setting) + '/' + str(run) + '/'
        
        
        path_to_interaction_train = path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0'
        path_to_interaction_val   = path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0'
        
        
        path_to_interaction_test  = path_to_run + f'interactions/{test_ds}/epoch_0/interaction_gpu0'
        
        interaction_train = torch.load(path_to_interaction_train, weights_only= False)
        interaction_val   = torch.load(path_to_interaction_val, weights_only= False)
        interaction_test  = torch.load(path_to_interaction_test, weights_only= False)
        
        
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val   = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        concepts_test  = objects_to_concepts(interaction_test.sender_input, n_values=n_values[i])
        
        messages_train = retrieve_messages(interaction_train)
        messages_val   = retrieve_messages(interaction_val)
        messages_test  = retrieve_messages(interaction_test)
        
        symbol_counts_train = count_symbols(messages_train)
        symbol_counts_val   = count_symbols(messages_val)
        symbol_counts_test  = count_symbols(messages_test)
        symbol_counts = [symbol_counts_train, symbol_counts_val, symbol_counts_test]
        
    
        pickle.dump(symbol_counts, open(path_to_run + f'symbol_counts_{zero_shot_test}.pkl', 'wb'))
        
        actual_vocab_size = len(symbol_counts_train + symbol_counts_val + symbol_counts_test)
        print(actual_vocab_size, "symbols used during training, validation, and testing.")
        pickle.dump(actual_vocab_size, open(path_to_run + f'vocab_size_{zero_shot_test}.pkl', 'wb'))
        
        # consider train and validation messages together
        messages_train_val = messages_train + messages_val
        # consider only unique messages
        messages_train_val_unique = get_unique_message_set(messages_train_val) 
        #print("messages train val", len(messages_train_val), len(messages_train_val_unique))
        messages_test_unique = get_unique_message_set(messages_test)
        #print("messages test", len(messages_test), len(messages_test_unique))
        # total messages
        messages_total = messages_train_val + messages_test
        messages_total_unique = get_unique_message_set(messages_total)
        
        # Concepts
        concepts_train_unique = get_unique_concept_set(concepts_train)
        concepts_val_unique   = get_unique_concept_set(concepts_val)
        concepts_test_unique  = get_unique_concept_set(concepts_test)
        concepts_total = concepts_train + concepts_val + concepts_test
        concepts_total_unique = get_unique_concept_set(concepts_total)
        
        num_of_concepts = [
            len(concepts_train_unique), 
            len(concepts_val_unique), 
            len(concepts_test_unique), 
            len(concepts_total_unique), 
            len(concepts_total)
        ]
        pickle.dump(num_of_concepts, open(path_to_run + f'num_of_concepts_{zero_shot_test}.pkl', 'wb'))
        
        # Messages reused in testing
        intersection = messages_train_val_unique & messages_test_unique
        # messages only used in training:
        difference_train = messages_train_val_unique - messages_test_unique
        # messages only used in testing:
        difference_test  = messages_test_unique - messages_train_val_unique
        
        print(len(difference_test), "novel messages used for the", len(concepts_test_unique), "novel concepts")
        
        message_reuse = [
            len(intersection),          
            len(difference_train),      
            len(difference_test),       
            len(concepts_test_unique),  
            len(difference_test)/len(concepts_test_unique),  
            len(messages_test_unique)   
        ]
        pickle.dump(message_reuse, open(path_to_run + f'message_reuse_{zero_shot_test}.pkl', 'wb'))


Dataset: 3dshapes
16 symbols used during training, validation, and testing.
44 novel messages used for the 1920 novel concepts
15 symbols used during training, validation, and testing.
83 novel messages used for the 1920 novel concepts
11 symbols used during training, validation, and testing.
41 novel messages used for the 1920 novel concepts


#### Results reported in the repo of zero-shot paper ("to specific" condition):

(3,4)
- 16 symbols from the alphabet have been actually used during training, validation and testing.
- 31 novel messages used for the 64 novel concepts
- 15 symbols from the alphabet have been actually used during training, validation and testing.
- 19 novel messages used for the 64 novel concepts
- 14 symbols from the alphabet have been actually used during training, validation and testing.
- 19 novel messages used for the 64 novel concepts
- 15 symbols from the alphabet have been actually used during training, validation and testing.
- 28 novel messages used for the 64 novel concepts
- 16 symbols from the alphabet have been actually used during training, validation and testing.
- 28 novel messages used for the 64 novel concepts

#### Vocab Size and Novelty Rate Comparison ("to specific" condition)

**Vocab Size:**
- In both datasets, agents use a similar range of symbols (between 14–16) from the available alphabet (16)
- This means that agents in both datasets use most of the available symbols, but the symbolic dataset output shows a more productive and adaptive use of the vocabulary when generalizing to new concepts

**Novelty Rate (Novel Messages per Novel Concepts):** 
  - In the 3dshapes dataset (44, 83, 41 novel messages for 1920 novel concepts across 3 runs) the ratio of novel messages to novel concepts is relatively low: only a small fraction of novel concepts are assigned unique messages in testing. This means less flexibility and less compositional generalization in the emergent language. Comparatively, the ratio of novel messages to novel concepts is much higher for (3,4) symbolic dataset, as a larger proportion of novel concepts receive unique messages (A range of 19–31 novel messages for 64 novel concepts across 5 runs). This means that agents are more likely to generate new, unique messages for new concepts. 


Overall, in the "to specific" condition, agents trained on the symbolic dataset are better at generating novel messages for novel concepts -> stronger compositionality and generalization. 
The 3dshapes agents tend to reuse messages more -> less robust generalization 

In [18]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'generic' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [19]:
# Determine vocab size and message reuse (3dshapes only: to generic)

# Loop through all 3dshapes datasets
for i, d in enumerate(datasets):
    print("Dataset:", d)
    
    for run in range(n_runs):
        
        path_to_run = paths[i] + '/' + str(setting) + '/' + str(run) + '/'
        
        
        path_to_interaction_train = path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0'
        path_to_interaction_val   = path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0'
        
        
        path_to_interaction_test  = path_to_run + f'interactions/{test_ds}/epoch_0/interaction_gpu0'
        
        interaction_train = torch.load(path_to_interaction_train, weights_only= False)
        interaction_val   = torch.load(path_to_interaction_val, weights_only= False)
        interaction_test  = torch.load(path_to_interaction_test, weights_only= False)
        
        
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val   = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        concepts_test  = objects_to_concepts(interaction_test.sender_input, n_values=n_values[i])
        
        messages_train = retrieve_messages(interaction_train)
        messages_val   = retrieve_messages(interaction_val)
        messages_test  = retrieve_messages(interaction_test)
        
        symbol_counts_train = count_symbols(messages_train)
        symbol_counts_val   = count_symbols(messages_val)
        symbol_counts_test  = count_symbols(messages_test)
        symbol_counts = [symbol_counts_train, symbol_counts_val, symbol_counts_test]
        
    
        pickle.dump(symbol_counts, open(path_to_run + f'symbol_counts_{zero_shot_test}.pkl', 'wb'))
        
        actual_vocab_size = len(symbol_counts_train + symbol_counts_val + symbol_counts_test)
        print(actual_vocab_size, "symbols used during training, validation, and testing.")
        pickle.dump(actual_vocab_size, open(path_to_run + f'vocab_size_{zero_shot_test}.pkl', 'wb'))
        
        # consider train and validation messages together
        messages_train_val = messages_train + messages_val
        # consider only unique messages
        messages_train_val_unique = get_unique_message_set(messages_train_val) 
        #print("messages train val", len(messages_train_val), len(messages_train_val_unique))
        messages_test_unique = get_unique_message_set(messages_test)
        #print("messages test", len(messages_test), len(messages_test_unique))
        # total messages
        messages_total = messages_train_val + messages_test
        messages_total_unique = get_unique_message_set(messages_total)
        
        # Concepts
        concepts_train_unique = get_unique_concept_set(concepts_train)
        concepts_val_unique   = get_unique_concept_set(concepts_val)
        concepts_test_unique  = get_unique_concept_set(concepts_test)
        concepts_total = concepts_train + concepts_val + concepts_test
        concepts_total_unique = get_unique_concept_set(concepts_total)
        
        num_of_concepts = [
            len(concepts_train_unique), 
            len(concepts_val_unique), 
            len(concepts_test_unique), 
            len(concepts_total_unique), 
            len(concepts_total)
        ]
        pickle.dump(num_of_concepts, open(path_to_run + f'num_of_concepts_{zero_shot_test}.pkl', 'wb'))
        
        # Messages reused in testing
        intersection = messages_train_val_unique & messages_test_unique
        # messages only used in training:
        difference_train = messages_train_val_unique - messages_test_unique
        # messages only used in testing:
        difference_test  = messages_test_unique - messages_train_val_unique
        
        print(len(difference_test), "novel messages used for the", len(concepts_test_unique), "novel concepts")
        
        message_reuse = [
            len(intersection),          
            len(difference_train),      
            len(difference_test),       
            len(concepts_test_unique),  
            len(difference_test)/len(concepts_test_unique),  
            len(messages_test_unique)   
        ]
        pickle.dump(message_reuse, open(path_to_run + f'message_reuse_{zero_shot_test}.pkl', 'wb'))


Dataset: 3dshapes
16 symbols used during training, validation, and testing.
0 novel messages used for the 600 novel concepts
16 symbols used during training, validation, and testing.
3 novel messages used for the 600 novel concepts
15 symbols used during training, validation, and testing.
0 novel messages used for the 600 novel concepts


#### Results reported in the repo main branch ("to generic" condition):
(3,4)
- 1 novel messages used for the 12 novel concepts
- 4 novel messages used for the 12 novel concepts
- 6 novel messages used for the 12 novel concepts
- 0 novel messages used for the 12 novel concepts
- 1 novel messages used for the 12 novel concepts

#### Vocab Size and Novelty Rate Comparison ("to generic" condition)

**Vocab Size:**
- In 3dshapes dataset, most of the available symbols (15–16) are used. This is most likely the case for the symbolic dataset as well, but to be specific, I did not find the printed output for vocab size for (3,4) symbolic

**Novelty Rate:**
  - 3dshapes Dataset: 0–3 novel messages for 600 novel concepts (across 3 runs)
  - (3,4) Symbolic Dataset: 0–6 novel messages for 12 novel concepts (across 5 runs)
  - Novelty rate is extremely low in both datasets: For 3dshapes, almost all novel concepts in the test set reuse messages seen during training/validation. This is also the case for symbolic dataset, with only a few novel messages being generated.

Overall, in the "to generic" condition, both datasets show low novelty rates, with agents reusing messages for novel concepts, which is in line with what is expected in this condition (from what was expected and subsequently reported in the zero-shot paper). 

### Summary Table: Message Reuse and Novelty Rate

In [13]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'specific' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [14]:
message_reuse_dict = {'intersection': [], 'difference train': [], 'difference test': [], 'concepts test unique': [], 'test ratio': [], 'messages test unique': [],
                      'reuse rate': [], 'novelty rate': [], 'total ratio': []}
for i, d in enumerate(datasets):
    intersection, train_difference, test_difference, test_concepts, test_ratio, test_messages, reuse_rate, novelty_rate, total_ratio = [], [], [], [], [], [], [], [], []
    for run in range(n_runs): 
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        message_reuse = pickle.load(open(path_to_run + 'message_reuse_' + str(test_ds) + '.pkl', 'rb'))
        intersection.append(message_reuse[0])
        train_difference.append(message_reuse[1])
        test_difference.append(message_reuse[2])
        test_concepts.append(message_reuse[3])
        test_ratio.append(message_reuse[4])
        test_messages.append(message_reuse[5])
        reuse_rate.append(message_reuse[0]/message_reuse[5])
        novelty_rate.append(message_reuse[2]/message_reuse[5])
        total_ratio.append(message_reuse[5]/message_reuse[3]) # test_messages / test_concepts (novel unique messages & concepts)

    message_reuse_dict['intersection'].append(intersection)
    message_reuse_dict['difference train'].append(train_difference)
    message_reuse_dict['difference test'].append(test_difference)
    message_reuse_dict['concepts test unique'].append(test_concepts)
    message_reuse_dict['test ratio'].append(test_ratio)
    message_reuse_dict['messages test unique'].append(test_messages)
    message_reuse_dict['reuse rate'].append(reuse_rate)
    message_reuse_dict['novelty rate'].append(novelty_rate)
    message_reuse_dict['total ratio'].append(total_ratio)

In [ ]:
message_reuse = [message_reuse_dict['concepts test unique'], message_reuse_dict['messages test unique'], message_reuse_dict['total ratio'], message_reuse_dict['reuse rate'], message_reuse_dict['novelty rate']]

# Convert the list to a NumPy array
mess_reuse_array = np.array(message_reuse)

# Compute means and standard deviations over the five runs
means = np.mean(mess_reuse_array, axis=-1)
std_devs = np.std(mess_reuse_array, axis=-1)

# Row names and column names
# row_names = ["D(3,4)", "D(3,8)", "D(3,16)", "D(4,4)", "D(4,8)", "D(5,4)"]
row_names = ["3dshapes"]
col_names = ["test concepts", "total unique messages", "message-concept ratio", "reuse rate","novelty rate"]

# Prepare the data for the DataFrames
data = []

# iterate over datasets
for i in range(means.shape[1]):
    row = []
    # iterate over conditions
    for j in range(means.shape[0]):
        if j > 1:
            formatted_value = f"{means[j, i]:.2f} $\\pm$ {std_devs[j, i]:.2f}"
        elif j == 0:
            formatted_value = f"{int(means[j, i])}"
        else:
            formatted_value = f"{means[j, i]:.1f} $\\pm$ {std_devs[j, i]:.1f}"
        row.append(formatted_value)
    data.append(row)

# # Create DataFrames
# df = pd.DataFrame(data, index=row_names, columns=col_names)

# # Convert DataFrames to LaTeX tables
# latex_table = df.to_latex(index=True, escape=False)
# print(df)
# print(latex_table)

# Create DataFrame
df = pd.DataFrame(data, index=row_names, columns=col_names)

# Save to CSV (easy to open in Excel or Google Sheets)
csv_path = "message_reuse_summary.csv"
df.to_csv(csv_path, index=True)

print(f"✅ Table saved as CSV at: {csv_path}")
### Results reported in the repo main branch ("to generic" condition):
(3,4)
- display(df)  # Nicely formatted output in Jupyter

✅ Table saved as CSV at: message_reuse_summary.csv


,test concepts,total unique messages,message-concept ratio,reuse rate,novelty rate
3dshapes,1920,125.7 $\pm$ 15.1,0.07 $\pm$ 0.01,0.56 $\pm$ 0.10,0.44 $\pm$ 0.10


In [16]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'generic' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [17]:
message_reuse_dict = {'intersection': [], 'difference train': [], 'difference test': [], 'concepts test unique': [], 'test ratio': [], 'messages test unique': [],
                      'reuse rate': [], 'novelty rate': [], 'total ratio': []}
for i, d in enumerate(datasets):
    intersection, train_difference, test_difference, test_concepts, test_ratio, test_messages, reuse_rate, novelty_rate, total_ratio = [], [], [], [], [], [], [], [], []
    for run in range(n_runs): 
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        message_reuse = pickle.load(open(path_to_run + 'message_reuse_' + str(test_ds) + '.pkl', 'rb'))
        intersection.append(message_reuse[0])
        train_difference.append(message_reuse[1])
        test_difference.append(message_reuse[2])
        test_concepts.append(message_reuse[3])
        test_ratio.append(message_reuse[4])
        test_messages.append(message_reuse[5])
        reuse_rate.append(message_reuse[0]/message_reuse[5])
        novelty_rate.append(message_reuse[2]/message_reuse[5])
        total_ratio.append(message_reuse[5]/message_reuse[3]) # test_messages / test_concepts (novel unique messages & concepts)

    message_reuse_dict['intersection'].append(intersection)
    message_reuse_dict['difference train'].append(train_difference)
    message_reuse_dict['difference test'].append(test_difference)
    message_reuse_dict['concepts test unique'].append(test_concepts)
    message_reuse_dict['test ratio'].append(test_ratio)
    message_reuse_dict['messages test unique'].append(test_messages)
    message_reuse_dict['reuse rate'].append(reuse_rate)
    message_reuse_dict['novelty rate'].append(novelty_rate)
    message_reuse_dict['total ratio'].append(total_ratio)

In [18]:
message_reuse = [message_reuse_dict['concepts test unique'], message_reuse_dict['messages test unique'], message_reuse_dict['total ratio'], message_reuse_dict['reuse rate'], message_reuse_dict['novelty rate']]

# Convert the list to a NumPy array
mess_reuse_array = np.array(message_reuse)

# Compute means and standard deviations over the five runs
means = np.mean(mess_reuse_array, axis=-1)
std_devs = np.std(mess_reuse_array, axis=-1)

# Row names and column names
# row_names = ["D(3,4)", "D(3,8)", "D(3,16)", "D(4,4)", "D(4,8)", "D(5,4)"]
row_names = ["3dshapes"]
col_names = ["test concepts", "total unique messages", "message-concept ratio", "reuse rate","novelty rate"]

# Prepare the data for the DataFrames
data = []

# iterate over datasets
for i in range(means.shape[1]):
    row = []
    # iterate over conditions
    for j in range(means.shape[0]):
        if j > 1:
            formatted_value = f"{means[j, i]:.2f} $\\pm$ {std_devs[j, i]:.2f}"
        elif j == 0:
            formatted_value = f"{int(means[j, i])}"
        else:
            formatted_value = f"{means[j, i]:.1f} $\\pm$ {std_devs[j, i]:.1f}"
        row.append(formatted_value)
    data.append(row)

# # Create DataFrames
# df = pd.DataFrame(data, index=row_names, columns=col_names)

# # Convert DataFrames to LaTeX tables
# latex_table = df.to_latex(index=True, escape=False)
# print(df)
# print(latex_table)

# Create DataFrame
df = pd.DataFrame(data, index=row_names, columns=col_names)

# Save to CSV (easy to open in Excel or Google Sheets)
csv_path = "message_reuse_summary.csv"
df.to_csv(csv_path, index=True)

print(f"✅ Table saved as CSV at: {csv_path}")
display(df)  # Nicely formatted output in Jupyter

✅ Table saved as CSV at: message_reuse_summary.csv


,test concepts,total unique messages,message-concept ratio,reuse rate,novelty rate
3dshapes,600,7.7 $\pm$ 4.2,0.01 $\pm$ 0.01,0.92 $\pm$ 0.12,0.08 $\pm$ 0.12


#### message_reuse_summary table reported in the zero-shot paper for D(3,4) symbolic dataset:
"to specific" condition
- reuse rate: 0.54 ± 0.07
- novelty rate: 0.46 ±0.07

"to generic" condition
- reuse rate: 0.80±0.19
- novelty rate: 0.20±0.19

#### Comparison of Message Reuse and Novelty Rates: 3dshapes vs. Symbolic D(3,4) Datasets

| Condition     | Dataset      | Reuse Rate         | Novelty Rate       |
|---------------|-------------|--------------------|--------------------|
| To Specific   | 3dshapes    | 0.56 ± 0.10        | 0.44 ± 0.10        |
|               | Symbolic    | 0.54 ± 0.07        | 0.46 ± 0.07        |
| To Generic    | 3dshapes    | 0.92 ± 0.12        | 0.08 ± 0.12        |
|               | Symbolic    | 0.80 ± 0.19        | 0.20 ± 0.19        |

**To Specific:**  
Both datasets show similar reuse and novelty rates in the "to specific" condition. The symbolic dataset has a slightly higher novelty rate, meaning it generates more unique messages for novel concepts compared to 3dshapes. 

**to generic:**  
In the "to generic" condition, both datasets show very high message reuse and very low novelty rates. The symbolic dataset has a slightly higher novelty rate than 3dshapes and the 3dshapes has a slightly higher reuse rate compared to the symbolic dataset. Overall, agents mostly reuse messages for new concepts in both datasets. 

Main takeaways:
- Symbolic D(3,4) consistently has a slightly higher novelty rate in both conditions, indicating it is somewhat better at generating new messages for new concepts.
- Agents trained on 3dshapes are more conservative, reusing messages more often, especially in the "to generic" condition.
- In the "to generic" condition, the symbolic dataset is marginally more flexible in compositional generalization which can be explained thorugh its structure: the discrete, well-defined concept space of symbolic dataset can facilitate message generation, while the structure of 3dshapes dataset may encourage more message reuse.

### Symbol reuse


**These are notes from the old version of the notebook:**
Also in "to generic" condition, all symbols are reused during testing, i.e. they all encode relevant information. This is why a qualitative analysis of messages makes more sense.
We need to compute, for each attribute-value (e.g., attribute 0 value 2), which symbol in the vocabulary is most informative (highest normalized mutual information) and the corresponding MI score

In [20]:
# symbol_frequency_MI is the correct one because objects[fixed == 1] = np.nan would mean erasing teh relavant attributes and basically does not make sence

def symbol_frequency_MI(interaction, n_attributes, n_values, vocab_size, is_gumbel=True):
    messages = interaction.message.argmax(dim=-1) if is_gumbel else interaction.message
    messages = messages[:, :-1]  # without EOS
    sender_input = interaction.sender_input
    n_objects = sender_input.shape[1]
    n_targets = int(n_objects / 2)
    target_objects = sender_input[:, :n_targets]
    target_objects = k_hot_to_attributes(target_objects, n_values)
    (objects, fixed) = retrieve_concepts_sampling(target_objects)

    # attributes which are not fixed are irrelevant to the concept and do not need to be communicated
    objects[fixed == 0] = np.nan

    favorite_symbol_MI = {}
    mutual_information = {}
    # find symbol with highest MI for each value at each attribute position (i.e., position sensitive)
    for att in range(n_attributes):
        for val in range(n_values):
            object_labels = (objects[:, att] == val).astype(int)
            max_MI = 0
            for symbol in range(vocab_size):
                symbol_indices = np.argwhere(messages == symbol)[0]
                symbol_labels = np.zeros(len(messages))
                symbol_labels[symbol_indices] = 1
                MI = normalized_mutual_info_score(symbol_labels, object_labels)
                if MI > max_MI:
                    max_MI = MI
                    max_symbol = symbol
            favorite_symbol_MI[str(att) + str(val)] = max_symbol
            mutual_information[str(att) + str(val)] = max_MI

    return favorite_symbol_MI, mutual_information

In [21]:
favorite_symbol_MI, mutual_information = symbol_frequency_MI(
    interaction_train,
    n_attributes=n_attributes[i],
    n_values=n_values[i],
    vocab_size=16,
)


print("Favorite symbols:")
for key, symbol in favorite_symbol_MI.items():
    print(f"Attribute-Value {key}: Symbol {symbol}")

    # results for this do not change across conditions- why?


Favorite symbols:
Attribute-Value 00: Symbol 5
Attribute-Value 01: Symbol 13
Attribute-Value 02: Symbol 2
Attribute-Value 03: Symbol 0
Attribute-Value 10: Symbol 14
Attribute-Value 11: Symbol 13
Attribute-Value 12: Symbol 0
Attribute-Value 13: Symbol 9
Attribute-Value 20: Symbol 3
Attribute-Value 21: Symbol 0
Attribute-Value 22: Symbol 13
Attribute-Value 23: Symbol 9


- In this output list, for each attribute-value pair, the symbol from the vocabulary that has the highest normalized mutual information (MI) with that attribute-value is printed.
- For example, "Attribute-Value 00: Symbol 5" means symbol 5 is most informative for attribute 0 having value 0.
- This mapping shows the emergent "lexicon" of the speaker agent:
    - If the mapping stays consistent and each attribute-value pair is associated with a distinct symbol -> it suggests a systematic and compositional language
    - If the same symbol is used for multiple attribute-value pairs -> it suggests less compositionality and more ambiguity in the emergent language

#### Symbol Reuse: 3dshapes, To specific Condition

In [ ]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'specific' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [ ]:
# this one is from the zero shot papaer repo, I tried adaptging it to the 3dshapes dataset, but did not succeed. 
# However, I printed out the cocepts to see how they look like, and to compare them with those from the symbolic dataset
# go through all datasets
for i, d in enumerate(datasets): # only 3dshapes here, to sprcific condition
    print(d)
    # get random qualitative samples
    # to specific: all indices should be fixed
    if zero_shot_test == 'specific':
        n_fixed = n_attributes[i]
        fixed_indices = list(range(0, n_attributes[i])) # all attributes fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define fixed values for these indices
    # to generic: only one index fixed, which one is randomly determined
    elif zero_shot_test == 'generic':
        n_fixed = 1
        fixed_indices = random.sample(range(0, n_attributes[i]), k=n_fixed) # select which attribute is fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define a fixed value for this index
    print(f"Fixed indices: {fixed_indices}, Fixed values: {fixed_values}")

    for run in range(n_runs):
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_test = (path_to_run + 'interactions/' + str(test_ds) + '/epoch_0/interaction_gpu0')
        interaction_train = torch.load(path_to_interaction_train, weights_only= False)
        interaction_val = torch.load(path_to_interaction_val, weights_only= False)
        interaction_test = torch.load(path_to_interaction_test, weights_only= False)
        
        # retrieve mapping between attribute-value pairs and symbols
        favorite_symbol, mutual_information = symbol_frequency_MI(interaction_train, n_attributes=n_attributes[i], n_values=n_values[i], vocab_size=16)
        print(favorite_symbol, mutual_information)
        
        messages = interaction_test.message.argmax(dim=-1)
        messages = [msg.tolist() for msg in messages]
        sender_input = interaction_test.sender_input
        n_targets = int(sender_input.shape[1]/2)
        # get target objects and fixed vectors to re-construct concepts
        target_objects = sender_input[:, :n_targets]
        target_objects = k_hot_to_attributes(target_objects, n_values[i])
        # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
        (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
        concepts = list(zip(objects, fixed))
        
        # get distractor objects to re-construct context conditions
        distractor_objects = sender_input[:, n_targets:]
        distractor_objects = k_hot_to_attributes(distractor_objects, n_values[i])
        context_conds = retrieve_context_condition(objects, fixed, distractor_objects)
        
        ##NEW# Extract concepts from interactions
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val   = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        concepts_test  = objects_to_concepts(interaction_test.sender_input, n_values=n_values[i])

        print(f"Dataset: {d}, Run: {run}") 
        print(f"Number of training concepts: {len(concepts_train)}") 
        print(f"Number of validation concepts: {len(concepts_val)}") 
        print(f"Number of test concepts: {len(concepts_test)}") 
        
        ##NEW# Extract messages
        messages_train = retrieve_messages(interaction_train)
        messages_val   = retrieve_messages(interaction_val)
        messages_test  = retrieve_messages(interaction_test)
        
        all_for_this_concept = []

        ##NEW# Select fixed indices from where fixed is 1
        available_indices = np.where(fixed == 1)[0]

        # Ensure available_indices only contains valid attribute indices
        valid_indices = list(range(n_attributes[i]))
        available_indices = [idx for idx in available_indices if idx in valid_indices]

        if len(available_indices) >= n_fixed:
            fixed_indices = random.sample(available_indices, k=n_fixed)
            fixed_values = random.choices(range(0, n_values[i]), k=n_fixed)
            print(f"Selected fixed_indices: {fixed_indices}, fixed_values: {fixed_values}")
        else:
            print(f"Not enough fixed indices available for concept {idx}")
            continue

        print(f"n_fixed: {n_fixed}, fixed_indices: {fixed_indices}, fixed_values: {fixed_values}")

        for idx, (t_objects, t_fixed) in enumerate(concepts_test[:3]):
            print(f"Concept {idx}: t_fixed = {t_fixed}, t_objects = {t_objects}")

            # Check if all fixed_indices are within the bounds of t_fixed
            if all(0 <= fixed_index < len(t_fixed) for fixed_index in fixed_indices):
                # Check if the sum of t_fixed at fixed_indices equals n_fixed
                if sum(t_fixed[fixed_indices]) == n_fixed:
                    for t_object in t_objects:
                        # Check if all t_object values at fixed_indices match fixed_values
                        if all(t_object[fixed_index] == fixed_values[j] for j, fixed_index in enumerate(fixed_indices)):
                            all_for_this_concept.append((idx, t_object, t_fixed, context_conds[idx], messages[idx]))
                            fixed = t_fixed
            else:
                print(f"Warning: fixed_indices out of bounds for t_fixed in concept {idx}")

        print(f"Total concepts matching criteria: {len(all_for_this_concept)}")
        if len(all_for_this_concept) > 0:
            print(f"\nSummary:")
            print(f"Messages used: {len(messages)}")
            print(f"Average message count: {np.mean(counts):.2f}")

            #sample = random.sample(all_for_this_concept, 20)
            sample = all_for_this_concept
            column_names = ['game_nr', 'object', 'fixed indices', 'context condition', 'message']
            sample_df = pd.DataFrame(sample, columns=column_names)
            # find out which messages have been used how often (once in test dataset test_sampled_unscaled)
            message_counts = sample_df.message.apply(tuple).value_counts()/10 # divide by game size because above single objects are taken, but we are interested in concepts (i.e. sets of game_size=10 objects)
            messages = message_counts.index.tolist()
            counts = message_counts.values.tolist()
            cond_indices = np.arange(0, len(messages)*10, 10)
            context_conds = sample_df['context condition'][cond_indices]
            used_symbols = look_up_values(fixed_indices, fixed_values, favorite_symbol)
            symbol_MI = look_up_values(fixed_indices, fixed_values, mutual_information)
            df_concept = pd.DataFrame({
                'fixed indices': [fixed_indices], 
                'fixed values': [fixed_values]
            })
            df_messages = pd.DataFrame({
                'context condition': context_conds.values.tolist(),
                'message': messages, 
                'counts': counts
            })
            df_symbols = pd.DataFrame({
                'symbols': used_symbols, 
                'symbol MI': symbol_MI
            })
            df = pd.concat([df_concept, df_messages, df_symbols])
            print(df.to_latex(float_format=lambda x: '{:,.0f}'.format(x) if x % 1 == 0 else '{:,.4f}'.format(x)))
            # df.to_csv('analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv', index=False)
            # print('saved ' + 'analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv')
        else:
            raise ValueError("sample for dataset " + str(d) + " could not be generated")

3dshapes
Fixed indices: [0, 1, 2], Fixed values: [1, 0, 0]
{'00': 0, '01': 3, '02': 9, '03': 0, '10': 0, '11': 3, '12': 0, '13': 8, '20': 13, '21': 0, '22': 14, '23': 8} {'00': 1.0, '01': 0.05679216549085574, '02': 0.08415974307741246, '03': 1.0, '10': 1.0, '11': 0.24252230486268547, '12': 1.0, '13': 0.531303768557855, '20': 0.13672419952246845, '21': 1.0, '22': 0.04373329483778888, '23': 0.48010420017741123}
Dataset: 3dshapes, Run: 0
Number of training concepts: 810
Number of validation concepts: 256
Number of test concepts: 1920
Selected fixed_indices: [2, 2, 2], fixed_values: [1, 3, 3]
n_fixed: 3, fixed_indices: [2, 2, 2], fixed_values: [1, 3, 3]
Concept 0: t_fixed = [0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 1. 0.
 1.], t_objects = [[0. 1. 3. 3. 1. 1. 3. 1. 2. 3. 3. 2. 0. 1. 1. 1. 1. 0. 0. 0. 0. 2. 3. 3.
  0.]
 [2. 3. 2. 3. 1. 1. 3. 1. 2. 3. 3. 2. 3. 2. 1. 3. 1. 3. 0. 0. 2. 2. 3. 2.
  0.]
 [0. 1. 3. 3. 1. 1. 3. 1. 2. 3. 3. 2. 0. 1. 1. 1. 1. 0. 0. 0. 0. 2. 3.

ValueError: sample for dataset 3dshapes could not be generated

#### Symbol Reuse: 3dshapes, To generic Condition

In [25]:
context_unaware = False # whether original or context_unaware simulations are evaluated
zero_shot = True # whether zero-shot simulations are evaluated
zero_shot_test = 'generic' # 'generic' or 'specific'
test_interactions = True # whether scores should be calculated on test interactions (only with zero shot)
test_ds = 'test' # 'test' or 'test_sampled_unscaled' or 'test_unscaled' or 'test_fine' 
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test

In [ ]:
# this one is from the zero shot papaer repo, I tried adaptging it to the 3dshapes dataset, but did not succeed. 
# However, I printed out the cocepts to see how they look like, and to compare them with those from the symbolic dataset
# go through all datasets
for i, d in enumerate(datasets): # only 3dshapes here, to generic condition
    print(d)
    # get random qualitative samples
    # to specific: all indices should be fixed
    if zero_shot_test == 'specific':
        n_fixed = n_attributes[i]
        fixed_indices = list(range(0, n_attributes[i])) # all attributes fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define fixed values for these indices
    # to generic: only one index fixed, which one is randomly determined
    elif zero_shot_test == 'generic':
        n_fixed = 1
        fixed_indices = random.sample(range(0, n_attributes[i]), k=n_fixed) # select which attribute is fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define a fixed value for this index
    print(f"Fixed indices: {fixed_indices}, Fixed values: {fixed_values}")

    for run in range(n_runs):
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_test = (path_to_run + 'interactions/' + str(test_ds) + '/epoch_0/interaction_gpu0')
        interaction_train = torch.load(path_to_interaction_train, weights_only= False)
        interaction_val = torch.load(path_to_interaction_val, weights_only= False)
        interaction_test = torch.load(path_to_interaction_test, weights_only= False)
        
        # retrieve mapping between attribute-value pairs and symbols
        favorite_symbol, mutual_information = symbol_frequency_MI(interaction_train, n_attributes=n_attributes[i], n_values=n_values[i], vocab_size=16)
        print(favorite_symbol, mutual_information)
        
        messages = interaction_test.message.argmax(dim=-1)
        messages = [msg.tolist() for msg in messages]
        sender_input = interaction_test.sender_input
        n_targets = int(sender_input.shape[1]/2)
        # get target objects and fixed vectors to re-construct concepts
        target_objects = sender_input[:, :n_targets]
        target_objects = k_hot_to_attributes(target_objects, n_values[i])
        # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
        (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
        concepts = list(zip(objects, fixed))
        
        # get distractor objects to re-construct context conditions
        distractor_objects = sender_input[:, n_targets:]
        distractor_objects = k_hot_to_attributes(distractor_objects, n_values[i])
        context_conds = retrieve_context_condition(objects, fixed, distractor_objects)
        
        ##NEW# Extract concepts from interactions
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val   = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        concepts_test  = objects_to_concepts(interaction_test.sender_input, n_values=n_values[i])

        print(f"Dataset: {d}, Run: {run}") 
        print(f"Number of training concepts: {len(concepts_train)}") 
        print(f"Number of validation concepts: {len(concepts_val)}") 
        print(f"Number of test concepts: {len(concepts_test)}") 
        
        ##NEW# Extract messages
        messages_train = retrieve_messages(interaction_train)
        messages_val   = retrieve_messages(interaction_val)
        messages_test  = retrieve_messages(interaction_test)
        
        all_for_this_concept = []

        ##NEW# Select fixed indices from where fixed is 1
        available_indices = np.where(fixed == 1)[0]

        # Ensure available_indices only contains valid attribute indices
        valid_indices = list(range(n_attributes[i]))
        available_indices = [idx for idx in available_indices if idx in valid_indices]

        if len(available_indices) >= n_fixed:
            fixed_indices = random.sample(available_indices, k=n_fixed)
            fixed_values = random.choices(range(0, n_values[i]), k=n_fixed)
            print(f"Selected fixed_indices: {fixed_indices}, fixed_values: {fixed_values}")
        else:
            print(f"Not enough fixed indices available for concept {idx}")
            continue

        print(f"n_fixed: {n_fixed}, fixed_indices: {fixed_indices}, fixed_values: {fixed_values}")

        for idx, (t_objects, t_fixed) in enumerate(concepts_test[:3]):
            print(f"Concept {idx}: t_fixed = {t_fixed}, t_objects = {t_objects}")

            # Check if all fixed_indices are within the bounds of t_fixed
            if all(0 <= fixed_index < len(t_fixed) for fixed_index in fixed_indices):
                # Check if the sum of t_fixed at fixed_indices equals n_fixed
                if sum(t_fixed[fixed_indices]) == n_fixed:
                    for t_object in t_objects:
                        # Check if all t_object values at fixed_indices match fixed_values
                        if all(t_object[fixed_index] == fixed_values[j] for j, fixed_index in enumerate(fixed_indices)):
                            all_for_this_concept.append((idx, t_object, t_fixed, context_conds[idx], messages[idx]))
                            fixed = t_fixed
            else:
                print(f"Warning: fixed_indices out of bounds for t_fixed in concept {idx}")

        print(f"Total concepts matching criteria: {len(all_for_this_concept)}")
        if len(all_for_this_concept) > 0:
            print(f"\nSummary:")
            print(f"Messages used: {len(messages)}")
            print(f"Average message count: {np.mean(counts):.2f}")

            #sample = random.sample(all_for_this_concept, 20)
            sample = all_for_this_concept
            column_names = ['game_nr', 'object', 'fixed indices', 'context condition', 'message']
            sample_df = pd.DataFrame(sample, columns=column_names)
            # find out which messages have been used how often (once in test dataset test_sampled_unscaled)
            message_counts = sample_df.message.apply(tuple).value_counts()/10 # divide by game size because above single objects are taken, but we are interested in concepts (i.e. sets of game_size=10 objects)
            messages = message_counts.index.tolist()
            counts = message_counts.values.tolist()
            cond_indices = np.arange(0, len(messages)*10, 10)
            context_conds = sample_df['context condition'][cond_indices]
            used_symbols = look_up_values(fixed_indices, fixed_values, favorite_symbol)
            symbol_MI = look_up_values(fixed_indices, fixed_values, mutual_information)
            df_concept = pd.DataFrame({
                'fixed indices': [fixed_indices], 
                'fixed values': [fixed_values]
            })
            df_messages = pd.DataFrame({
                'context condition': context_conds.values.tolist(),
                'message': messages, 
                'counts': counts
            })
            df_symbols = pd.DataFrame({
                'symbols': used_symbols, 
                'symbol MI': symbol_MI
            })
            df = pd.concat([df_concept, df_messages, df_symbols])
            print(df.to_latex(float_format=lambda x: '{:,.0f}'.format(x) if x % 1 == 0 else '{:,.4f}'.format(x)))
            # df.to_csv('analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv', index=False)
            # print('saved ' + 'analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv')
        else:
            raise ValueError("sample for dataset " + str(d) + " could not be generated")

3dshapes
Fixed indices: [1], Fixed values: [2]
{'00': 4, '01': 10, '02': 2, '03': 0, '10': 0, '11': 10, '12': 2, '13': 8, '20': 12, '21': 0, '22': 10, '23': 7} {'00': 0.004186360734832301, '01': 0.034663892401478144, '02': 0.23263140077195418, '03': 1.0, '10': 1.0, '11': 0.5250541014309882, '12': 0.008015786997930392, '13': 0.2922747359843243, '20': 0.31161353797464464, '21': 1.0, '22': 0.29129112499216736, '23': 0.22417593136439548}
Dataset: 3dshapes, Run: 0
Number of training concepts: 2160
Number of validation concepts: 704
Number of test concepts: 600
Selected fixed_indices: [2], fixed_values: [2]
n_fixed: 1, fixed_indices: [2], fixed_values: [2]
Concept 0: t_fixed = [0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1.
 1.], t_objects = [[1. 3. 0. 1. 1. 0. 3. 3. 3. 3. 1. 1. 0. 1. 2. 3. 0. 1. 3. 2. 1. 3. 2. 3.
  1.]
 [3. 3. 0. 1. 1. 1. 0. 1. 3. 3. 1. 0. 2. 1. 2. 2. 0. 1. 3. 2. 0. 3. 2. 3.
  1.]
 [3. 3. 0. 1. 1. 2. 0. 1. 3. 0. 1. 0. 2. 2. 0. 2. 2. 1. 3. 3. 0. 3. 0

ValueError: sample for dataset 3dshapes could not be generated

Next, I plan to print out the concepts communicated by the agent trained on symbolic dataset and compare the output of symbol reuse cells from the 3dshapes to that of symbolic dataset. By doing so, we can see the inner structural differences between the concepts communicated in each set of dataset.  

#### Symbol Reuse: D(3,4) Symbolic Dataset, To Specific Condition

In [27]:

### Configurations
datasets = ['(3,4)']  # Symbolic dataset
n_values = [4]
n_attributes = [3]
n_epochs = 300
n_runs = 3
n_datasets = len(datasets)

paths = [
    'results/(3,4)_game_size_10_vsf_3'  # path for symbolic (3,4)
]

context_unaware = False
zero_shot = True  # No zero-shot analysis
zero_shot_test = 'specific'
test_interactions = False  # Only train/val
test_ds = 'test'
setting = ""
if context_unaware:
    setting = setting + 'context_unaware'
else:
    setting = setting + 'standard'
if zero_shot:
    setting = setting + '/zero_shot/' + zero_shot_test


In [ ]:
# the one from the zero shot papaer repo
# # go through all datasets
for i, d in enumerate(datasets):
    print(d)
    # get random qualitative samples
    # to specific: all indices should be fixed
    if zero_shot_test == 'specific':
        n_fixed = n_attributes[i]
        fixed_indices = list(range(0, n_attributes[i])) # all attributes fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define fixed values for these indices
    # to generic: only one index fixed, which one is randomly determined
    elif zero_shot_test == 'generic':
        n_fixed = 1
        fixed_indices = random.sample(range(0, n_attributes[i]), k=n_fixed) # select which attribute is fixed
        fixed_values = random.choices(range(0, n_values[i]), k=n_fixed) # define a fixed value for this index
    print(fixed_indices, fixed_values)

    for run in range(n_runs):
        path_to_run = paths[i] + '/' + str(setting) +'/' + str(run) + '/'
        path_to_interaction_train = (path_to_run + 'interactions/train/epoch_' + str(n_epochs) + '/interaction_gpu0')
        path_to_interaction_val = (path_to_run + 'interactions/validation/epoch_' + str(n_epochs) + '/interaction_gpu0')
        #path_to_interaction_test = (path_to_run + 'interactions/' + str(test_ds) + '/epoch_0/interaction_gpu0') #REMOVED: I do not have the test interactions
        interaction_train = torch.load(path_to_interaction_train, weights_only= False)
        interaction_val = torch.load(path_to_interaction_val, weights_only= False)
        #interaction_test = torch.load(path_to_interaction_test, weights_only= False) #REMOVED: I do not have the test interactions
        
        # retrieve mapping between attribute-value pairs and symbols
        favorite_symbol, mutual_information = symbol_frequency_MI(interaction_train, n_attributes=n_attributes[i], n_values=n_values[i], vocab_size=16)
        print(favorite_symbol, mutual_information)
        
        #messages = interaction_test.message.argmax(dim=-1) #REMOVED
        #messages = [msg.tolist() for msg in messages] #REMOVED
        sender_input = interaction_train.sender_input #CHANGED from interaction_test to interaction_train
        n_targets = int(sender_input.shape[1]/2)
        # get target objects and fixed vectors to re-construct concepts
        target_objects = sender_input[:, :n_targets]
        target_objects = k_hot_to_attributes(target_objects, n_values[i])
        # concepts are defined by a list of target objects (here one sampled target object) and a fixed vector
        (objects, fixed) = retrieve_concepts_sampling(target_objects, all_targets=True)
        concepts = list(zip(objects, fixed))

        
        
        # get distractor objects to re-construct context conditions
        distractor_objects = sender_input[:, n_targets:]
        distractor_objects = k_hot_to_attributes(distractor_objects, n_values[i])
        context_conds = retrieve_context_condition(objects, fixed, distractor_objects)
        
        #New:Extract concepts from interactions
        concepts_train = objects_to_concepts(interaction_train.sender_input, n_values=n_values[i])
        concepts_val   = objects_to_concepts(interaction_val.sender_input, n_values=n_values[i])
        

        print(f"Dataset: {d}, Run: {run}") # ADDED
        print(f"Number of training concepts: {len(concepts_train)}") # ADDED
        print(f"Number of validation concepts: {len(concepts_val)}") # ADDED
        
        #New:Extract messages
        messages_train = retrieve_messages(interaction_train)
        messages_val   = retrieve_messages(interaction_val)
        
        all_for_this_concept = []

        # NEW: Select fixed indices from where t_fixed is 1
        available_indices = np.where(fixed == 1)[0]  # CHANGED from t_fixed to fixed
        # Ensure available_indices only contains valid attribute indices
        valid_indices = list(range(n_attributes[i]))
        available_indices = [idx for idx in available_indices if idx in valid_indices]

        if len(available_indices) >= n_fixed: 
            fixed_indices = random.sample(list(available_indices), k=n_fixed)
            fixed_values = random.choices(range(0, n_values[i]), k=n_fixed)
            print(f"Selected fixed_indices: {fixed_indices}, fixed_values: {fixed_values}")  # Debugging
        else:
            print(f"Not enough fixed indices available for concept {idx}")
            continue  # Skip this concept

        print(f"n_fixed: {n_fixed}, fixed_indices: {fixed_indices}, fixed_values: {fixed_values}") # ADDED

        for idx, (t_objects, t_fixed) in enumerate(concepts_train): # concepts CHANGED to concepts_train
            print(f"Concept {idx}: t_fixed = {t_fixed}, t_objects = {t_objects}") # ADDED
            #if sum(t_fixed) == n_fixed and all(t_fixed[fixed_index] == 1 for fixed_index in fixed_indices): #OLD
            if sum(t_fixed[fixed_indices]) == n_fixed: #NEW
                for t_object in t_objects:
                    if all(t_object[fixed_index] == fixed_values[j] for j, fixed_index in enumerate(fixed_indices)):
                        all_for_this_concept.append((idx, t_object, t_fixed, context_conds[idx], messages_train[idx])) #CHANGED messages to messages_train
                        fixed = t_fixed
        print(f"Number of concepts matching criteria: {len(all_for_this_concept)}") # ADDED
        if len(all_for_this_concept) > 0:
            #sample = random.sample(all_for_this_concept, 20)
            sample = all_for_this_concept
            column_names = ['game_nr', 'object', 'fixed indices', 'context condition', 'message']
            sample_df = pd.DataFrame(sample, columns=column_names)
            # find out which messages have been used how often (once in test dataset test_sampled_unscaled)
            message_counts = sample_df.message.apply(tuple).value_counts()/10 # divide by game size because above single objects are taken, but we are interested in concepts (i.e. sets of game_size=10 objects)
            messages = message_counts.index.tolist()
            counts = message_counts.values.tolist()
            cond_indices = np.arange(0, len(messages)*10, 10)
            context_conds = sample_df['context condition'][cond_indices]
            used_symbols = look_up_values(fixed_indices, fixed_values, favorite_symbol)
            symbol_MI = look_up_values(fixed_indices, fixed_values, mutual_information)
            df_concept = pd.DataFrame({
                'fixed indices': [fixed_indices], 
                'fixed values': [fixed_values]
            })
            df_messages = pd.DataFrame({
                'context condition': context_conds.values.tolist(),
                'message': messages, 
                'counts': counts
            })
            df_symbols = pd.DataFrame({
                'symbols': used_symbols, 
                'symbol MI': symbol_MI
            })
            df = pd.concat([df_concept, df_messages, df_symbols])
            print(df.to_latex(float_format=lambda x: '{:,.0f}'.format(x) if x % 1 == 0 else '{:,.4f}'.format(x)))
            # df.to_csv('analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv', index=False)
            # print('saved ' + 'analysis/' + str(zero_shot_test) + '/quali_' + str(d) + '_' + str(zero_shot_test) + '_' + str(run) + '_' + str(fixed_indices) + ',' + str(fixed_values) + 'message_symbol_counts.csv')
        else:
            raise ValueError("sample for dataset " + str(d) + " could not be generated")

(3,4)
[0, 1, 2] [0, 3, 1]
{'00': 1, '01': 4, '02': 7, '03': 3, '10': 10, '11': 14, '12': 15, '13': 8, '20': 13, '21': 6, '22': 11, '23': 3} {'00': 0.4293127131615817, '01': 0.7340246766822486, '02': 0.4253535467014733, '03': 0.2506840201655253, '10': 0.9811174800754168, '11': 1.0, '12': 0.39721247331354514, '13': 1.0, '20': 0.4722874103343677, '21': 0.43193365269461675, '22': 0.746470485342744, '23': 0.26374951835055216}
Dataset: (3,4), Run: 0
Number of training concepts: 810
Number of validation concepts: 270
Selected fixed_indices: [2, 0, 0], fixed_values: [0, 0, 0]
n_fixed: 3, fixed_indices: [2, 0, 0], fixed_values: [0, 0, 0]
Concept 0: t_fixed = [1. 0. 1.], t_objects = [[1. 0. 2.]
 [1. 3. 2.]
 [1. 0. 2.]
 [1. 1. 2.]
 [1. 0. 2.]
 [1. 2. 2.]
 [1. 2. 2.]
 [1. 1. 2.]
 [1. 2. 2.]
 [1. 0. 2.]]
Concept 1: t_fixed = [1. 0. 1.], t_objects = [[0. 2. 2.]
 [0. 2. 2.]
 [0. 1. 2.]
 [0. 1. 2.]
 [0. 0. 2.]
 [0. 3. 2.]
 [0. 0. 2.]
 [0. 1. 2.]
 [0. 2. 2.]
 [0. 2. 2.]]
Concept 2: t_fixed = [0. 1. 1.]

-------------------------------------------------

#### Comparison and Conclusion:

- **3dshapes** 
    - t_fixed has a different structure in concepts communicated by the agent trained 3dshapes dataset: It seems to me that t_fixed is not a one-hot vector indicating which attributes are fixed. Instead, it appears to be a binary mask with containing a mix of 0s and 1s. This may mean that the stzructure of t-fixed in 3dshapes dataset allows for several attributes to be fixed at the same time?!
    - Addionally, the structure and encoding of t_fixed is way more complex: I expected to see the same structure as what we see with the symbolic dataset D(3,4), such as [1. 0. 1.], particularly because we define the same number of attributes and values in configs for both datasets. It seems to be the case that in 3dshapes, the encoding is more complex, both in t_fixed and t_objects.
    - On the other hand, t_objects contains attribute values as expected: These values seem to range from 0 to 3, consistent with n_values = 4. Nevertheless, the structure of t_objects in 3dshapes is multi-dimensional compared to that of t_objects in the symbolic dataset. 
    - This could all be due to a different type of encoding for the 3dshapes dataset, which seems to not follow the direct representation of indices as it is the case in D(3,4). 

- **What to do? What is the problem? (Warning: This is not coming from a programmer)**
    - The code needs to be adapted to handle the fact that t_fixed can encode fixed attributes using a mix of 0s and 1s. I believe that the current logic assumes a single fixed attribute and only follows that via use of fixed indices which cannot be possible applied to concepts in 3dshapes dataset.
    - The issue could be coming from how idices are being filtered out to only include valid attribute indices. 